In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv("/kaggle/input/titanic/train.csv")

In [ ]:
df

In [ ]:
df['Ticket'].unique()

In [ ]:
df['Cabin'].nunique()

In [ ]:
df['Fare'].nunique()

In [ ]:
df.nunique()

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df.describe()

In [ ]:
df['Embarked'].nunique()

## We will start the feature engneering 
I bleive that passengerid , name , ticket , cabin won't help us alot due to the huge variation in the data 

In [ ]:
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

In [ ]:
df['Age'].isna().sum()


In [ ]:
df['Fare'].isna().sum()

## Both fare and Age does not contain any NaN values we will peocede with Minmax scaling since both are pretty important numerical feutures

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])


## One hot encoding for Sex , Embarked , Pclass , SibSp and Parch

In [ ]:
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Pclass', 'SibSp', 'Parch'])


In [ ]:
df

In [ ]:
df.nunique()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()
df['Age'] = df['Age'].fillna(df['Age'].mean())

In [ ]:
df.isna().sum()

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=5000),
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)


In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring='accuracy'
)

grid_rf.fit(X_train, y_train)


In [ ]:
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

grid_svm = GridSearchCV(
    SVC(probability=True),
    param_grid_svm,
    cv=5,
    scoring='accuracy'
)

grid_svm.fit(X_train, y_train)


In [ ]:
models = {
    "Logistic Regression": grid_lr,
    "Random Forest": grid_rf,
    "SVM": grid_svm
}

for name, model in models.items():
    print(name, "Best Score:", model.best_score_)


In [ ]:
best_model = grid_rf.best_estimator_  # example


In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(
    best_model,
    X,
    y,
    cv=5,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)

test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure()

# Training curve
plt.plot(train_sizes, train_mean)
plt.fill_between(train_sizes,
                 train_mean - train_std,
                 train_mean + train_std,
                 alpha=0.2)

# Validation curve
plt.plot(train_sizes, test_mean)
plt.fill_between(train_sizes,
                 test_mean - test_std,
                 test_mean + test_std,
                 alpha=0.2)

plt.xlabel("Training Size")
plt.ylabel("Accuracy")
plt.title("Learning Curve")
plt.legend(["Training Score", "Validation Score"])
plt.show()


In [ ]:
import pandas as pd

def create_submission_gbt(model, test_df, submission_filename, X_train_columns, scaler, age_mean):
    """
    Creates a Kaggle submission CSV for Titanic using a trained model
    with exact preprocessing as training data.
    
    Parameters:
    - model: trained model (e.g., best_gbt)
    - test_df: raw test dataset
    - submission_filename: output CSV filename
    - X_train_columns: training features after preprocessing
    - scaler: fitted MinMaxScaler for Age and Fare
    - age_mean: mean Age from training set (for imputation)
    """

    passenger_ids = test_df['PassengerId']

    test_proc = test_df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

    test_proc['Age'] = test_proc['Age'].fillna(age_mean)

    test_proc[['Age', 'Fare']] = scaler.transform(test_proc[['Age', 'Fare']])

    features = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked']
    X_test = pd.get_dummies(test_proc[features])


    X_test = X_test.reindex(columns=X_train_columns, fill_value=0)

    predictions = model.predict(X_test)

  
    submission = pd.DataFrame({
        'PassengerId': passenger_ids,
        'Survived': predictions
    })

  
    submission.to_csv(submission_filename, index=False)
    print(f"Submission saved to {submission_filename}!")


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier


gbt_model = HistGradientBoostingClassifier(
    max_iter=500,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

gbt_model.fit(X_train, y_train)


y_pred = gbt_model.predict(X_test)
y_proba = gbt_model.predict_proba(X_test)[:, 1]  # for metrics like ROC-AUC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_gbt = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_iter': [200, 500]
}

grid_gbt = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    param_grid_gbt,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_gbt.fit(X_train, y_train)

best_gbt = grid_gbt.best_estimator_
print("Best parameters:", grid_gbt.best_params_)


In [ ]:
best_gbt

In [ ]:
create_submission_gbt(
    model=best_gbt,
    test_df=test_data,
    submission_filename='submission.csv',
    X_train_columns=X_train.columns,
    scaler=scaler,
    age_mean=X_train['Age'].mean()
)
